**Questões**
1. Qual configuração do menu lateral levou mais usuários a acessarem a página específica?
2. Qual o tempo médio de sessão para cada configuração do menu?
3. Qual é a taxa de conversão para cada configuração do menu?
4. A diferença nos acessos entre as configurações é estatisticamente significativa?
5. Quais insights podem ser derivados dos dados para otimizar a navegação do site e aumentar o engajamento dos usuários?
6. Existe alguma tendência temporal (comportamento ao longo dia) nos acessos ou conversões para cada configuração do menu?
7. Quais recomendações você faria com base nos dados analisados para otimizar a navegação do site?

## Importando bibliotecas e requerimentos

A seguir são importadas as bibliotecas [] utilizadas neste documento. A linha %pip


In [ ]:
#%pip install -r requirements.txt -q

import pandas as pd
from ipydatagrid import DataGrid
#from itables import show
from scipy.stats import chi2_contingency

dados = pd.read_excel("ab_test_data.xlsx")

dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 2305 entries, 0 to 2304
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           2305 non-null   int64         
 1   device            2305 non-null   str           
 2   country           2305 non-null   str           
 3   timestamp         2305 non-null   datetime64[us]
 4   group             2305 non-null   str           
 5   converted         2305 non-null   int64         
 6   session_duration  2269 non-null   float64       
 7   hour_of_day       2305 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(3), str(3)
memory usage: 144.2 KB


In [22]:
# 1. Garantindo que a linha do tempo está correta (ordem cronológica)
dados_cronologicos = dados.sort_values(by=['user_id', 'timestamp'])

# 2. Aplicando a sua ideia: O primeiro acesso dita as regras, mas olhamos o todo para a conversão
dados_consolidados = dados_cronologicos.groupby('user_id').agg(
    group=('group', 'first'),              # Pega o grupo do primeiro acesso
    country=('country', 'first'),          # Pega o país do primeiro acesso
    device=('device', 'first'),            # Pega o dispositivo do primeiro acesso
    converted=('converted', 'max'),        # True se converteu em QUALQUER momento (1 > 0)
    session_duration=('session_duration', 'mean') # Média do tempo válido (ignora NaNs)
).reset_index()

print(f"Base consolidada: {len(dados_consolidados)} usuários únicos prontos para análise.")
display(dados_consolidados.head())

Base consolidada: 1295 usuários únicos prontos para análise.


,user_id,group,country,device,converted,session_duration
0,1,Nível 0,EUA,Tablet,0,0.000000
1,2,Nível 1,Reino Unido,Smartphone,0,0.000000
2,4,Nível 0,Índia,Smartphone,0,0.000000
3,5,Nível 0,França,Tablet,1,9.702414
4,6,Nível 1,França,Smartphone,0,0.000000


In [21]:
#display(DataGrid(dados))

from ipydatagrid import DataGrid, TextRenderer, Expr

formatacao = {
    # 'Cor' if cell.value == True else 'Outra Cor'
    "converted": TextRenderer(
        background_color=Expr("'#d4edda' if cell.value == True else '#f8d7da'"),
        text_color=Expr("'#155724' if cell.value == True else '#721c24'")
    ),
    # Valores nulos do Pandas (NaN) são convertidos para None pelo ipydatagrid
    "session_duration": TextRenderer(
        background_color=Expr("'#fff3cd' if cell.value == None else 'white'")
    ),
    "group": TextRenderer(
        background_color="#e2e3e5",
        font="bold 12px Arial"
    )
}

# 2. Aplicando o estilo na tabela
grade_estilizada = DataGrid(
    dados, 
    selection_mode="row", 
    editable=False,
    base_row_size=35,           
    base_column_header_size=40, 
    renderers=formatacao        
)

print("Tabela formatada: Valores nulos em amarelo e conversões destacadas.")
display(grade_estilizada)

Tabela formatada: Valores nulos em amarelo e conversões destacadas.


DataGrid(auto_fit_params={'area': 'all', 'padding': 30, 'numCols': None}, base_column_header_size=40, base_row…

## Limpeza e tratamento

A coluna `hour_of_day` será removida por redundância (informação existente em `timestamp`);

A coluna `converted` será convertida para o formato booleano.

Foram identificados 36 nulos na coluna `session_duration`, representando provável erro de coleta (cerca de 1.6% do total). 
Os valores serão removidos apenas para análises de duração, demais análises usam o dataset completo.

In [ ]:
# Ordenando a base cronologicamente para garantir que o primeiro acesso venha primeiro
dados_ordenados = dados.sort_values(by=['user_id', 'timestamp'])

# Agregando os dados a nível de usuário (1 linha por usuário)
dados_usuarios = dados_ordenados.groupby('user_id').agg(
    group=('group', 'first'),              # Mantém o grupo que ele viu no PRIMEIRO acesso
    country=('country', 'first'),          # Mantém o país do PRIMEIRO acesso
    device=('device', 'first'),            # Mantém o dispositivo do PRIMEIRO acesso
    converteu=('converted', 'max'),        # Se converteu em QUALQUER sessão, conta como True (1)
    tempo_medio_sessao=('session_duration', 'mean'), # Média de tempo gasto
    acessos_totais=('timestamp', 'count')  # Quantas vezes ele acessou no total
).reset_index()

print(f"📊 Base transformada: {len(dados_usuarios)} usuários únicos.")
display(dados_usuarios.head())

In [ ]:
# 1. Identificando os IDs que possuem pelo menos uma sessão com duração nula (NaN)
ids_com_nulo = dados[dados['session_duration'].isna()]['user_id'].unique()

# 2. Resgatando TODAS as sessões (nulas e não nulas) apenas desses usuários
analise_nulos = dados[dados['user_id'].isin(ids_com_nulo)].copy()

# 3. Ordenando por usuário e timestamp para construir a "linha do tempo" de cada um
analise_nulos = analise_nulos.sort_values(by=['user_id', 'timestamp'])

print(f"Encontramos {len(ids_com_nulo)} usuários que tiveram erro de coleta no tempo de sessão.")
print("Explore a tabela abaixo para comparar as sessões falhas com as sessões normais do mesmo usuário:")

# 4. Usando o DataGrid para você poder brincar com os filtros visuais
from ipydatagrid import DataGrid

grade_nulos = DataGrid(analise_nulos, selection_mode="row", editable=False)
display(grade_nulos)

In [ ]:
dados["converted"] = dados["converted"].astype(bool)

dados = dados.drop(columns=["hour_of_day"])

dados.info()
